In [10]:
# Cell 1 (Corrected): Install stable dependencies
# Use a supported TensorFlow version, such as the lowest stable one listed in the error.
!pip install --quiet speechbrain soundfile librosa fastapi uvicorn[standard] python-multipart aiofiles scikit-learn requests joblib
!pip install --quiet "tensorflow==2.17.1" "numpy==1.24.3"

# Note: If 2.16.1 causes issues, try 2.17.1 (as originally attempted in your notebook)
# or the lowest stable version currently supported on the Colab environment.

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [3]:
import numpy as np
import tensorflow as tf
print(np.__version__)
print(tf.__version__)

2.0.2
2.19.0


In [4]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive', force_remount=True)

ROOT = Path('/content/drive/MyDrive/Speaker_Recognition_System')
FEATURES = ROOT / "data" / "features"   # where Notebook-02 saved X_embeddings.npy, y_labels.npy or manifest
META = ROOT / "metadata"
MODELS = ROOT / "models"
SERVER = ROOT / "server"

for p in [FEATURES, META, MODELS, SERVER]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)


Mounted at /content/drive
ROOT: /content/drive/MyDrive/Speaker_Recognition_System


In [5]:
import numpy as np
# Adjust if your files are X_embeddings.npy / y_labels.npy
X_path = FEATURES / "X_embeddings.npy"
y_path = FEATURES / "y_labels.npy"

X = np.load(X_path)
y = np.load(y_path)

# If embeddings are (N,T,D) average across time
if X.ndim == 3:
    print("Pooling time-dimension")
    X = X.mean(axis=1)

print("Embeddings shape:", X.shape, "Labels:", np.unique(y, return_counts=True))


Embeddings shape: (496, 192) Labels: (array([0, 1]), array([486,  10]))


In [12]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# scale
sc = StandardScaler().fit(X_train)
X_train_s = sc.transform(X_train)
X_test_s = sc.transform(X_test)

# build model
inp_dim = X_train_s.shape[1]
model = tf.keras.Sequential([
    tf.keras.Input(shape=(inp_dim,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# train
model.fit(X_train_s, y_train, validation_split=0.1, epochs=25, batch_size=32)

# eval
loss, acc = model.evaluate(X_test_s, y_test, verbose=0)
print("Test acc:", acc)

# save Keras model and scaler
# Save in H5 format first
model.save(MODELS / "tf_classifier.h5")

import joblib
joblib.dump(sc, MODELS / "scaler.save")
print("Saved tf_classifier.h5 and scaler")

# Now convert the H5 model to SavedModel format for TFLite conversion
# This will be done in the next cell.

Epoch 1/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 220ms/step - accuracy: 0.6725 - loss: 0.6689 - val_accuracy: 1.0000 - val_loss: 0.0293
Epoch 2/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9702 - loss: 0.1020 - val_accuracy: 1.0000 - val_loss: 0.0056
Epoch 3/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9743 - loss: 0.0432 - val_accuracy: 1.0000 - val_loss: 0.0033
Epoch 4/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0122 - val_accuracy: 1.0000 - val_loss: 0.0027
Epoch 5/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0126 - val_accuracy: 1.0000 - val_loss: 0.0023
Epoch 6/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0063 - val_accuracy: 1.0000 - val_loss: 0.0018
Epoch 7/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0048 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 8/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0047 - val_accuracy: 1.0000 - val_los

Test acc: 0.9919354915618896
Saved tf_classifier.h5 and scaler


In [13]:
# Float32 TFLite
import tensorflow as tf
# Use the SavedModel directory for conversion
converter = tf.lite.TFLiteConverter.from_saved_model(str(MODELS/"tf_classifier_savedmodel"))
tflite_model = converter.convert()
open(MODELS/"model.tflite","wb").write(tflite_model)
print("Saved float32 model.tflite")

# Optional: INT8 quantization (requires representative dataset)
def representative_gen():
    for i in range(min(300, X_train_s.shape[0])):
        yield [X_train_s[i].astype('float32').reshape(1, -1)]

converter = tf.lite.TFLiteConverter.from_saved_model(str(MODELS/"tf_classifier_savedmodel"))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_quant = converter.convert()
open(MODELS/"model_int8.tflite","wb").write(tflite_quant)
print("Saved int8 model_int8.tflite (useful for mobile)")

OSError: SavedModel file does not exist at: /content/drive/MyDrive/Speaker_Recognition_System/models/tf_classifier_savedmodel/{saved_model.pbtxt|saved_model.pb}

In [14]:
%%bash
cat > app.py <<'PY'
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
import json, io, numpy as np, soundfile as sf, torch
from pathlib import Path
from speechbrain.pretrained import EncoderClassifier
from sklearn.metrics.pairwise import cosine_similarity
import joblib

# load ECAPA (server-side)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ECAPA = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="ecapa_server", run_opts={"device":DEVICE})

# server DB paths
ROOT = Path("speaker_server")
ROOT.mkdir(exist_ok=True)
DB_PATH = ROOT / "speaker_db.json"
if DB_PATH.exists():
    speaker_db = json.load(open(DB_PATH))
else:
    speaker_db = {}  # {name: [prototype_vector]}

# load tf classifier for fallback (server-side)
SCALER = None
TF_MODEL = None
try:
    SCALER = joblib.load("models/scaler.save")
    import tensorflow as tf
    TF_MODEL = tf.keras.models.load_model("models/tf_classifier.keras")
except Exception:
    pass

app = FastAPI(title="Speaker Verification - Hybrid")

def get_emb_from_bytes(data_bytes):
    wav = io.BytesIO(data_bytes)
    y, sr = sf.read(wav, dtype='float32')
    if y.ndim > 1: y = y.mean(axis=1)
    # remove silence lightly (optional)
    # convert to torch and get embedding
    wav_t = torch.tensor(y).unsqueeze(0)
    with torch.no_grad():
        emb = ECAPA.encode_batch(wav_t).squeeze().cpu().numpy()
    emb = emb / (np.linalg.norm(emb) + 1e-12)
    return emb

@app.post("/enroll")
async def enroll(name: str = Form(...), file: UploadFile = File(...)):
    data = await file.read()
    emb = get_emb_from_bytes(data).tolist()
    # store prototype (single sample for now; could average multiple)
    speaker_db[name] = {"prototype": emb}
    json.dump(speaker_db, open(DB_PATH, "w"), indent=2)
    return {"status":"ok", "message":f"Enrolled {name}"}

@app.post("/verify")
async def verify(file: UploadFile = File(...)):
    if not speaker_db:
        return JSONResponse({"status":"error","error":"No enrolled speakers"}, status_code=400)
    data = await file.read()
    emb = get_emb_from_bytes(data).reshape(1, -1)

    # 1) prototype cosine check (fast)
    best = (None, -1.0)
    for name, info in speaker_db.items():
        proto = np.array(info["prototype"]).reshape(1, -1)
        sim = float(cosine_similarity(emb, proto)[0][0])
        if sim > best[1]: best = (name, sim)

    # load threshold metrics saved by Notebook-04, fallback to 0.60
    try:
        metrics = json.load(open("metadata/security_metrics.json"))
        threshold = float(metrics.get("THRESHOLD", 0.60))
    except Exception:
        threshold = 0.60

    if best[1] >= threshold:
        return {"status":"ok", "decision":"FAMILIAR", "best":best[0], "score":best[1], "via":"prototype"}
    else:
        # 2) fallback classifier (if available)
        if TF_MODEL is not None and SCALER is not None:
            x = SCALER.transform(emb)  # shape (1, D)
            probs = TF_MODEL.predict(x)[0]
            pred = int(np.argmax(probs))
            conf = float(np.max(probs))
            return {"status":"ok", "decision":"FAMILIAR" if pred==1 else "STRANGER", "score":conf, "via":"classifier"}
        else:
            return {"status":"ok", "decision":"STRANGER", "best":best[0], "score":best[1], "via":"prototype"}

@app.get("/speakers")
def speakers():
    return {"speakers": list(speaker_db.keys())}
PY
echo "app.py created"


app.py created


In [15]:
# Not recommended in production. For quick local dev testing only.
import subprocess, sys, time
proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "info"])
print("uvicorn started, PID:", proc.pid)
# Keep track of proc to terminate later with proc.terminate()


uvicorn started, PID: 5751


In [16]:
# Use these to test when server is running locally. Replace URL when deployed.
import requests

server = "http://127.0.0.1:8000"

# Enroll example
def enroll_local(name, wav_path):
    r = requests.post(f"{server}/enroll", files={"file": open(wav_path,"rb")}, data={"name":name}, timeout=60)
    print(r.status_code, r.text)

# Verify example
def verify_local(wav_path):
    r = requests.post(f"{server}/verify", files={"file": open(wav_path,"rb")}, timeout=60)
    print(r.status_code, r.json())

# Example usage:
# enroll_local("Parent1", "/content/drive/MyDrive/Speaker_Recognition_System/data/raw/familiar_01.wav")
# verify_local("/content/drive/MyDrive/Speaker_Recognition_System/data/raw/test.wav")


In [17]:
# TFLite files are in MODELS folder (model.tflite and model_int8.tflite)
print("TFLite models saved to:", MODELS)
print("Download model.tflite (float) or model_int8.tflite (quant) to Android app assets or internal storage.")


TFLite models saved to: /content/drive/MyDrive/Speaker_Recognition_System/models
Download model.tflite (float) or model_int8.tflite (quant) to Android app assets or internal storage.


In [21]:
# dependencies: implementation 'com.squareup.okhttp3:okhttp:4.11.0', implementation 'org.jetbrains.kotlinx:kotlinx-coroutines-android:1.7.3'

import okhttp3.*
import java.io.File
import kotlinx.coroutines.Dispatchers
import kotlinx.coroutines.withContext

suspend fun postWavToServer(wavFile: File, serverUrl: String): String {
    return withContext(Dispatchers.IO) {
        val client = OkHttpClient()
        val MEDIA_TYPE_WAV = "audio/wav".toMediaTypeOrNull()
        val reqBody = MultipartBody.Builder()
            .setType(MultipartBody.FORM)
            .addFormDataPart("file", wavFile.name, wavFile.asRequestBody(MEDIA_TYPE_WAV))
            .build()
        val request = Request.Builder().url(serverUrl).post(reqBody).build()
        client.newCall(request).execute().use { resp ->
            if (!resp.isSuccessful) throw Exception("HTTP ${resp.code}")
            resp.body?.string() ?: ""
        }
    }
}


SyntaxError: invalid syntax (ipython-input-3790016153.py, line 3)

In [22]:
# Use TensorFlow Lite Interpreter (implementation 'org.tensorflow:tensorflow-lite:2.14.0')
# Load model.tflite and run input embedding through it (embedding must be same scaled as training)

val interpreter = Interpreter(File(filesDir, "model.tflite"))

# assume embedding: FloatArray of size D (apply same StandardScaler transform on device - store mean+std)
val input = arrayOf(embeddingFloatArray)             // shape [1, D]
val output = Array(1) { FloatArray(2) }              // 2 classes
interpreter.run(input, output)
val probs = output[0]
val predicted = probs.indices.maxByOrNull { probs[it] } ?: 0


SyntaxError: invalid syntax (ipython-input-3925485181.py, line 4)